# Phase 3: Triple Fusion (Image + Text + Clinical Metadata)
**PneumoFusionNet** — Multimodal Pneumonia Detection

Architecture: DenseNet-121+CBAM (Image) + ClinicalBERT (Text) + Clinical MLP (Metadata)


In [1]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings, json, copy, re, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_curve)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torchxrayvision as xrv
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
# --- CELL 1: Configuration ---
BASE_DIR     = r'c:\\2026\\PneumoFusionNet\\mimic\\1000_dataset'
CLINICAL_CSV = os.path.join(BASE_DIR, 'phase3_clinical_data.csv')
REPORT_CSV   = os.path.join(BASE_DIR, 'phase2_reports_no_impression.csv')
BBOX_CSV     = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_crossval', 'lung_bboxes.csv')
P1_CKPT      = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_crossval', 'best_model_fold2.pth')
SAVE_DIR     = os.path.join(BASE_DIR, 'outputs', 'Phase_3_triple_fusion')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model
CLINBERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
IMG_SIZE       = 224
MAX_TEXT_LEN   = 256
BATCH_SIZE     = 16
EPOCHS         = 40
PATIENCE       = 8
LR_FUSION      = 2e-4
LR_BERT        = 1e-5
LR_META        = 1e-3
FOCAL_GAMMA    = 2.0
MIXUP_ALPHA    = 0.2
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.15
SEED           = 42

# Clinical features to use
CLINICAL_FEATURES = [
    'age', 'gender_M',          # Demographics
    'is_deceased',
    'heart_rate', 'respiratory_rate', 'spo2',  # Vitals
    'systolic_bp', 'diastolic_bp', 'temperature_f',
    'wbc', 'hemoglobin', 'hematocrit',         # Labs
    'creatinine', 'crp', 'alk_phos', 'albumin'
]
NUM_CLINICAL = len(CLINICAL_FEATURES)  # 16 features
META_EMBED_DIM = 64  # Clinical metadata embedding dimension

print(f'Clinical features: {NUM_CLINICAL}')
print(f'Save dir: {SAVE_DIR}')

Clinical features: 16
Save dir: c:\\2026\\PneumoFusionNet\\mimic\\1000_dataset\outputs\Phase_3_triple_fusion


In [3]:
# --- CELL 2: Data Loading & Clinical Feature Engineering ---
# Load clinical data
clinical_df = pd.read_csv(CLINICAL_CSV)
print(f'Clinical CSV: {clinical_df.shape}')

# Load reports (for text)
reports_df = pd.read_csv(REPORT_CSV)
print(f'Reports CSV: {reports_df.shape}')

# Merge: add report text to clinical data
df = clinical_df.merge(
    reports_df[['subject_id', 'study_id', 'report_no_impression']],
    on=['subject_id', 'study_id'],
    how='left'
)
df = df.dropna(subset=['label'])
print(f'Merged: {df.shape}')

# --- Feature Engineering ---
# One-hot encode gender
df['gender_M'] = (df['gender'] == 'M').astype(float)

# Fill missing vitals/labs with median
for col in CLINICAL_FEATURES:
    if col in df.columns and col != 'gender_M':
        median_val = df[col].median()
        missing = df[col].isna().sum()
        df[col].fillna(median_val, inplace=True)
        if missing > 0:
            print(f'  {col}: filled {missing} missing with median={median_val:.2f}')

# Normalise all clinical features to 0-1 range
clinical_stats = {}
for col in CLINICAL_FEATURES:
    mn, mx = df[col].min(), df[col].max()
    if mx > mn:
        df[col] = (df[col] - mn) / (mx - mn)
    clinical_stats[col] = {'min': float(mn), 'max': float(mx)}
    print(f'  {col}: range [{mn:.2f}, {mx:.2f}] -> [0, 1]')

# Resolve image paths
def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\\\', os.sep)
    return os.path.join(BASE_DIR, p) if not os.path.isabs(p) else p

df['image_path'] = df['image_path'].apply(resolve_path)
df = df[df['image_path'].apply(lambda x: os.path.exists(x) if x else False)]
print(f'\\nFinal dataset after path check: {len(df)}')
print(f'  Normal: {(df["label"]==0).sum()}, Pneumonia: {(df["label"]==1).sum()}')

Clinical CSV: (3221, 24)
Reports CSV: (1989, 10)
Merged: (3425, 25)
  age: filled 150 missing with median=57.00
  is_deceased: filled 150 missing with median=0.00
  heart_rate: filled 2080 missing with median=84.00
  respiratory_rate: filled 2080 missing with median=19.00
  spo2: filled 2080 missing with median=97.00
  systolic_bp: filled 2085 missing with median=118.50
  diastolic_bp: filled 2085 missing with median=64.00
  temperature_f: filled 2084 missing with median=98.30
  wbc: filled 205 missing with median=7.50
  hemoglobin: filled 205 missing with median=12.05
  hematocrit: filled 205 missing with median=36.70
  creatinine: filled 205 missing with median=0.90
  crp: filled 214 missing with median=26.00
  alk_phos: filled 214 missing with median=14.00
  albumin: filled 742 missing with median=4.05
  age: range [18.00, 91.00] -> [0, 1]
  gender_M: range [0.00, 1.00] -> [0, 1]
  is_deceased: range [0.00, 1.00] -> [0, 1]
  heart_rate: range [35.00, 145.00] -> [0, 1]
  respiratory_

In [4]:
# --- CELL 3: Improved Text Extraction (FINDINGS + HISTORY) ---
LEAKAGE_RE = re.compile(
    r'\bpneumonia\b|\bpneumonic\b|\bno[ -]acute\b|\bnormal[ -]study\b|'
    r'\bno[ -]finding\b|\bnormal[ -]chest\b|\bunremarkable\b',
    re.IGNORECASE
)

def extract_rich_text(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return '[NO REPORT]'
    raw = str(raw)
    parts = []
    # Extract HISTORY
    hist_match = re.search(r'HISTORY[:\s]+(.*?)(?=TECHNIQUE|COMPARISON|FINDINGS|$)', raw, re.S|re.I)
    if hist_match:
        parts.append('HISTORY: ' + hist_match.group(1).strip()[:200])
    # Extract FINDINGS
    find_match = re.search(r'FINDINGS[:\s]+(.*?)(?=IMPRESSION|$)', raw, re.S|re.I)
    if find_match:
        parts.append('FINDINGS: ' + find_match.group(1).strip())
    text = ' '.join(parts) if parts else raw[:500]
    text = LEAKAGE_RE.sub('[REDACTED]', text)
    text = re.sub(r'_{2,}', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 10 else '[NO REPORT]'

df['report_clean'] = df['report_no_impression'].apply(extract_rich_text)
print(f'Reports with content: {(df["report_clean"] != "[NO REPORT]").sum()} / {len(df)}')
print(f'\\nSample NORMAL text:\\n  {df[df["label"]==0]["report_clean"].iloc[0][:150]}')
print(f'\\nSample PNEUMONIA text:\\n  {df[df["label"]==1]["report_clean"].iloc[0][:150]}')

Reports with content: 2653 / 3425
\nSample NORMAL text:\n  HISTORY: Fever, dyspnea and cough. Question left basilar [REDACTED]. FINDINGS: The heart appears again mildly enlarged. The mediastinal and hilar cont
\nSample PNEUMONIA text:\n  [NO REPORT]


In [5]:
# --- CELL 4: Train / Val / Test Split (70/15/15, stratified) ---
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

for name, sub in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:5s}:  {len(sub)}  (Normal={int((sub["label"]==0).sum())}, Pneumonia={int((sub["label"]==1).sum())})')

ValueError: Input y contains NaN.

In [ ]:
# --- CELL 5: Triple-Modal Dataset Class ---
bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup)}')

class TripleModalDataset(Dataset):
    def __init__(self, dataframe, tokenizer, transform=None, max_len=256):
        self.df        = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def _load_image(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
        # CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = clahe.apply(img)
        # Bbox crop
        key = path.replace('\\\\', '/')
        if key in bbox_lookup:
            b = bbox_lookup[key]
            x,y,w,h = int(b['x']), int(b['y']), int(b['w']), int(b['h'])
            img = img[max(0,y):y+h, max(0,x):x+w]
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0
        img = np.stack([img]*3, axis=0)  # 3-channel
        img = torch.tensor(img, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Image
        img = self._load_image(row['image_path'])
        # Text
        enc = self.tokenizer(
            row['report_clean'], padding='max_length',
            truncation=True, max_length=self.max_len, return_tensors='pt')
        ids  = enc['input_ids'].squeeze(0)
        mask = enc['attention_mask'].squeeze(0)
        # Clinical metadata vector
        meta = torch.tensor([row[f] for f in CLINICAL_FEATURES], dtype=torch.float32)
        # Label
        label = torch.tensor(int(row['label']), dtype=torch.long)
        return img, ids, mask, meta, label

print('Dataset class ready.')

In [ ]:
# --- CELL 6: Model Architecture ---

# ---- CBAM (same as Phase 1 & 2) ----
class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.mx=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Conv2d(c,c//r,1,bias=False),nn.ReLU(),nn.Conv2d(c//r,c,1,bias=False))
    def forward(self,x): return x*torch.sigmoid(self.fc(self.avg(x))+self.fc(self.mx(x)))

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__()
        self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False)
    def forward(self,x):
        return x*torch.sigmoid(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1)))

class CBAM(nn.Module):
    def __init__(self,c,r=16):
        super().__init__(); self.ca=ChannelAttention(c,r); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

# ---- Image Encoder (Frozen, from Phase 1) ----
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features = base.features
        self.cbam     = CBAM(1024, r=16)
        self.pool     = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        f = self.features(x)
        f = self.cbam(f)
        return self.pool(f).flatten(1)  # (B, 1024)

# ---- Text Encoder (ClinicalBERT, last 2 layers unfrozen) ----
class TextEncoder(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # Freeze all then unfreeze last 2 layers
        for p in self.bert.parameters():
            p.requires_grad = False
        for layer in [self.bert.encoder.layer[10], self.bert.encoder.layer[11]]:
            for p in layer.parameters():
                p.requires_grad = True
    def forward(self, ids, mask):
        out = self.bert(input_ids=ids, attention_mask=mask)
        return out.last_hidden_state  # (B, seq_len, 768)

# ---- Clinical Metadata Encoder ----
class MetadataEncoder(nn.Module):
    def __init__(self, num_features, embed_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )
    def forward(self, x):
        return self.mlp(x)  # (B, embed_dim)

# ---- Triple Fusion with Cross-Attention ----
class TripleFusionNet(nn.Module):
    def __init__(self, img_dim=1024, txt_dim=768, meta_dim=64, hidden=512, n_heads=8):
        super().__init__()
        # Project image and text to same dimension
        self.img_proj  = nn.Linear(img_dim, hidden)
        self.txt_proj  = nn.Linear(txt_dim, hidden)
        # Cross-attention: image queries text
        self.cross_attn = nn.MultiheadAttention(hidden, n_heads, dropout=0.1, batch_first=True)
        self.attn_norm  = nn.LayerNorm(hidden)
        # Fusion head: cross_attn_out + metadata
        self.classifier = nn.Sequential(
            nn.Linear(hidden + meta_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, img_feat, txt_seq, txt_mask, meta_feat):
        # Project
        img_q = self.img_proj(img_feat).unsqueeze(1)        # (B,1,H)
        txt_kv = self.txt_proj(txt_seq)                      # (B,seq,H)
        # Cross-attention
        key_pad_mask = (txt_mask == 0)                        # True = padded
        attn_out, _ = self.cross_attn(img_q, txt_kv, txt_kv, key_padding_mask=key_pad_mask)
        fused = self.attn_norm(attn_out.squeeze(1))          # (B, H)
        # Concat with metadata
        combined = torch.cat([fused, meta_feat], dim=1)       # (B, H+meta_dim)
        return self.classifier(combined)

print('All model classes defined.')
print(f'Triple Fusion: Image(1024) + Text(768) + Metadata({NUM_CLINICAL}) -> Cross-Attn -> Predict')

In [ ]:
# --- CELL 7: Focal Loss & Mixup Utilities ---

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

def mixup_data(img, txt_seq, txt_mask, meta, labels, alpha=0.2):
    if alpha <= 0:
        return img, txt_seq, txt_mask, meta, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    idx = torch.randperm(img.size(0)).to(img.device)
    img_mix  = lam * img  + (1 - lam) * img[idx]
    meta_mix = lam * meta + (1 - lam) * meta[idx]
    return img_mix, txt_seq, txt_mask, meta_mix, labels, labels[idx], lam

print('Focal Loss & Mixup ready.')

In [ ]:
# --- CELL 8: Instantiate Models, Tokenizer & DataLoaders ---
print('Loading ClinicalBERT tokenizer...')
tokenizer    = AutoTokenizer.from_pretrained(CLINBERT_MODEL)
text_encoder = TextEncoder(CLINBERT_MODEL).to(DEVICE)
print('  ClinicalBERT loaded (last 2 layers unfrozen)')

print('Loading Phase 1 image encoder...')
img_encoder  = ImageEncoder().to(DEVICE)
ckpt = torch.load(P1_CKPT, map_location=DEVICE)
loaded = img_encoder.load_state_dict(ckpt, strict=False)
print(f'  [ImageEncoder] loaded {len(ckpt)} keys | missing={len(loaded.missing_keys)} | unexpected={len(loaded.unexpected_keys)}')
# Freeze image encoder
for p in img_encoder.parameters():
    p.requires_grad = False
img_encoder.eval()

print('Creating metadata encoder...')
meta_encoder = MetadataEncoder(NUM_CLINICAL, META_EMBED_DIM).to(DEVICE)
print(f'  MetadataEncoder: {NUM_CLINICAL} features -> {META_EMBED_DIM}-d')

print('Creating triple fusion model...')
fusion_model = TripleFusionNet(img_dim=1024, txt_dim=768, meta_dim=META_EMBED_DIM).to(DEVICE)

# DataLoaders
train_tfm = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
])

train_ds = TripleModalDataset(train_df, tokenizer, transform=train_tfm, max_len=MAX_TEXT_LEN)
val_ds   = TripleModalDataset(val_df,   tokenizer, max_len=MAX_TEXT_LEN)
test_ds  = TripleModalDataset(test_df,  tokenizer, max_len=MAX_TEXT_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

# Class weights for focal loss
n0 = (train_df['label']==0).sum(); n1 = (train_df['label']==1).sum()
w = torch.tensor([1.0, n0/n1], dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=w)
print(f'Class weights: Normal=1.00, Pneumonia={n0/n1:.2f}')

In [ ]:
# --- CELL 9: Training Loop ---

def eval_epoch(fusion, img_enc, txt_enc, meta_enc, loader, criterion, device):
    fusion.eval(); txt_enc.eval(); meta_enc.eval()
    total_loss, all_probs, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, ids, masks, metas, labels in loader:
            imgs, ids, masks, metas, labels = (
                imgs.to(device), ids.to(device), masks.to(device),
                metas.to(device), labels.to(device))
            img_feat = img_enc(imgs)
            txt_seq  = txt_enc(ids, masks)
            meta_emb = meta_enc(metas)
            logits   = fusion(img_feat, txt_seq, masks, meta_emb)
            total_loss += criterion(logits, labels).item() * len(labels)
            all_probs.extend(F.softmax(logits, 1)[:,1].cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, (np.array(all_probs)>0.5).astype(int))
    auc = roc_auc_score(all_labels, all_probs)
    return avg_loss, acc, auc, np.array(all_probs), np.array(all_labels)

# Separate parameter groups with different LRs
param_groups = [
    {'params': fusion_model.parameters(),  'lr': LR_FUSION},
    {'params': [p for p in text_encoder.parameters() if p.requires_grad], 'lr': LR_BERT},
    {'params': meta_encoder.parameters(),  'lr': LR_META},
]
optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc, best_state, patience_cnt = 0.0, None, 0
history = {'train_loss':[],'val_loss':[],'train_auc':[],'val_auc':[],'lr':[]}

t0 = time.time()
for epoch in range(1, EPOCHS+1):
    fusion_model.train(); text_encoder.train(); meta_encoder.train()
    ep_loss, ep_probs, ep_labels = 0, [], []

    for imgs, ids, masks, metas, labels in train_loader:
        imgs, ids, masks, metas, labels = (
            imgs.to(DEVICE), ids.to(DEVICE), masks.to(DEVICE),
            metas.to(DEVICE), labels.to(DEVICE))

        img_feat = img_encoder(imgs)
        
        # Mixup on image features and metadata
        img_feat_m, _, _, metas_m, labels_a, labels_b, lam = mixup_data(
            img_feat, ids, masks, metas, labels, MIXUP_ALPHA)
        
        txt_seq  = text_encoder(ids, masks)
        meta_emb = meta_encoder(metas_m)
        logits   = fusion_model(img_feat_m, txt_seq, masks, meta_emb)
        
        loss = lam * criterion(logits, labels_a) + (1-lam) * criterion(logits, labels_b)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(fusion_model.parameters()) + 
            list(text_encoder.parameters()) +
            list(meta_encoder.parameters()), 1.0)
        optimizer.step()

        ep_loss += loss.item() * len(labels)
        ep_probs.extend(F.softmax(logits.detach(),1)[:,1].cpu().numpy())
        ep_labels.extend(labels_a.cpu().numpy())

    scheduler.step()
    tr_loss = ep_loss / len(train_loader.dataset)
    tr_auc  = roc_auc_score(ep_labels, ep_probs)

    val_loss, val_acc, val_auc, _, _ = eval_epoch(
        fusion_model, img_encoder, text_encoder, meta_encoder,
        val_loader, criterion, DEVICE)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc)
    history['val_auc'].append(val_auc)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    elapsed = time.time() - t0
    spe = elapsed / epoch
    eta = spe * (EPOCHS - epoch)

    marker = ''
    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {
            'fusion': copy.deepcopy(fusion_model.state_dict()),
            'text':   copy.deepcopy(text_encoder.state_dict()),
            'meta':   copy.deepcopy(meta_encoder.state_dict()),
        }
        patience_cnt = 0
        marker = ' <-- best'
        torch.save(best_state, os.path.join(SAVE_DIR, 'best_phase3_model.pth'))
    else:
        patience_cnt += 1

    print(f'Ep {epoch:03d}/{EPOCHS} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} | '
          f'val_auc={val_auc:.4f} val_acc={val_acc:.3f} | '
          f'lr={optimizer.param_groups[0]["lr"]:.1e} | {spe:.0f}s/ep | ETA:{eta/60:.0f}m{marker}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
# --- CELL 10: Test Set Evaluation ---
fusion_model.load_state_dict(best_state['fusion'])
text_encoder.load_state_dict(best_state['text'])
meta_encoder.load_state_dict(best_state['meta'])

_, test_acc, test_auc, test_probs, test_labels = eval_epoch(
    fusion_model, img_encoder, text_encoder, meta_encoder,
    test_loader, criterion, DEVICE)

print(f'Test AUC: {test_auc:.4f}')

# Youden-J threshold
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
youden_thresh = thresholds[best_j_idx]
print(f'Youden-J threshold: {youden_thresh:.4f}')

# Metrics at different thresholds
for name, th in [('Default (0.5)', 0.5), ('Youden-J', youden_thresh)]:
    preds = (test_probs >= th).astype(int)
    acc = accuracy_score(test_labels, preds)
    cm = confusion_matrix(test_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp+fn) if (tp+fn)>0 else 0
    spec = tn / (tn+fp) if (tn+fp)>0 else 0
    print(f'{name:20s}: Acc={acc:.4f} Sens={sens:.4f} Spec={spec:.4f}')

In [ ]:
# --- CELL 11: Training Curves ---
BG='#0f1117'; AX='#1a1d27'; TC='#ecf0f1'; GC='#2c3e50'
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor(BG)
for ax,(k1,k2),title in zip(axes,[('train_loss','val_loss'),('train_auc','val_auc')],['Loss','AUC']):
    ax.set_facecolor(AX)
    ax.plot(history[k1], label='Train', color='#3498db', linewidth=2)
    ax.plot(history[k2], label='Val',   color='#e74c3c', linewidth=2)
    ax.set_title(f'Phase 3 -- {title}', color=TC, fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch', color=TC); ax.set_ylabel(title, color=TC)
    ax.legend(facecolor=AX, edgecolor=GC, labelcolor=TC)
    ax.tick_params(colors=TC); [s.set_color(GC) for s in ax.spines.values()]
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'v3_training_curves.png'), dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved training curves.')

In [ ]:
# --- CELL 12: Confusion Matrices ---
BG='#0f1117'; TC='#ecf0f1'

fig, axes = plt.subplots(1,2,figsize=(14,5.5))
fig.patch.set_facecolor(BG)
fig.suptitle(f'Phase 3 Triple Fusion -- Confusion Matrices (Test N={len(test_labels)})',
             color=TC, fontsize=15, fontweight='bold', y=1.02)

cmaps = ['Blues', 'Greens']
thresh_list = [('Default (0.50)', 0.5), ('Youden-J', youden_thresh)]

for ax, (tname, th), cmap in zip(axes, thresh_list, cmaps):
    preds = (test_probs >= th).astype(int)
    cm = confusion_matrix(test_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    acc  = (tp+tn)/cm.sum()

    ax.set_facecolor('#1a1d27')
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Normal','Pneumonia'], yticklabels=['Normal','Pneumonia'],
                annot_kws={'size':18, 'fontweight':'bold'}, linewidths=2, linecolor='#0f1117')
    ax.set_title(f'{tname} ({th:.3f})', color=TC, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', color=TC, fontsize=11)
    ax.set_ylabel('Actual', color=TC, fontsize=11)
    ax.tick_params(colors=TC)
    caption = f'Sens: {sens*100:.1f}% | Spec: {spec*100:.1f}% | Acc: {acc*100:.1f}%'
    ax.text(1.0, -0.12, caption, transform=ax.transAxes, ha='center',
            color=TC, fontsize=10, fontstyle='italic')

fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'v3_confusion_matrices.png'), dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved confusion matrices.')

In [ ]:
# --- CELL 13: ROC Curve ---
BG='#0f1117'; AX='#1a1d27'; TC='#ecf0f1'; GC='#2c3e50'
fpr, tpr, _ = roc_curve(test_labels, test_probs)
fig, ax = plt.subplots(figsize=(8,7))
fig.patch.set_facecolor(BG); ax.set_facecolor(AX)
ax.plot(fpr, tpr, color='#2ecc71', linewidth=2.5, label=f'Phase 3 AUC = {test_auc:.4f}')
ax.plot([0,1],[0,1],'--', color='#7f8c8d', linewidth=1)
ax.scatter([1-((test_probs>=youden_thresh).astype(int)==0).sum()/((test_labels==0).sum())],
           [((test_probs>=youden_thresh).astype(int)==1)[test_labels==1].sum()/((test_labels==1).sum())],
           s=150, c='gold', zorder=5, label=f'Youden-J (t={youden_thresh:.3f})')
ax.set_xlabel('False Positive Rate', color=TC, fontsize=12)
ax.set_ylabel('True Positive Rate', color=TC, fontsize=12)
ax.set_title(f'Phase 3 Triple Fusion -- ROC Curve', color=TC, fontsize=14, fontweight='bold')
ax.legend(facecolor=AX, edgecolor=GC, labelcolor=TC, fontsize=11)
ax.tick_params(colors=TC); [s.set_color(GC) for s in ax.spines.values()]
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'v3_roc_curve.png'), dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()

In [ ]:
# --- CELL 14: Phase 1 vs Phase 2 vs Phase 2v2 vs Phase 3 Comparison ---
BG='#0f1117'; AX='#1a1d27'; TC='#ecf0f1'; GC='#2c3e50'

# Load previous results
p1_json = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1v4_PA_crossval', 'cv_results.json')
p2_json = os.path.join(BASE_DIR, 'outputs', 'Phase_2_multimodal', 'phase2_results.json')
p2v2_json = os.path.join(BASE_DIR, 'outputs', 'Phase_2v2_improved', 'phase2v2_results.json')

p1 = json.load(open(p1_json))
p2 = json.load(open(p2_json))
p2v2 = json.load(open(p2v2_json))

# Current Phase 3 results
preds_y = (test_probs >= youden_thresh).astype(int)
cm_y = confusion_matrix(test_labels, preds_y)
tn,fp,fn,tp = cm_y.ravel()
p3_sens = tp/(tp+fn)
p3_spec = tn/(tn+fp)
p3_acc  = (tp+tn)/cm_y.sum()

models = ['Phase 1\n(Image)', 'Phase 2 v1\n(Img+Txt)', 'Phase 2 v2\n(Cross-Attn)', 'Phase 3\n(Triple)']
aucs   = [p1['fold_details'][1]['tta_auc'], p2['phase2_test_auc'], p2v2['test_auc'], test_auc]
accs   = [p1['fold_details'][1]['tuned_acc'], p2['phase2_tuned_acc'], p2v2['youden_acc'], p3_acc]
senss  = [p1['fold_details'][1]['pn_recall'], p2['phase2_sensitivity'], p2v2['youden_sensitivity'], p3_sens]

fig, axes = plt.subplots(1,3, figsize=(18,6))
fig.patch.set_facecolor(BG)
fig.suptitle('PneumoFusionNet: Evolution Across All Phases', color=TC, fontsize=16, fontweight='bold', y=1.02)

colors = ['#3498db', '#9b59b6', '#e74c3c', '#2ecc71']
for ax, data, title in zip(axes, [aucs, accs, senss], ['AUC', 'Accuracy', 'Sensitivity']):
    ax.set_facecolor(AX)
    bars = ax.bar(models, data, color=colors, width=0.6, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', color=TC, fontweight='bold', fontsize=12)
    ax.set_title(title, color=TC, fontsize=14, fontweight='bold')
    ax.set_ylim(0.65, 1.0)
    ax.tick_params(colors=TC); [s.set_color(GC) for s in ax.spines.values()]

fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'all_phases_comparison.png'), dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved comparison chart.')

In [ ]:
# --- CELL 15: Clinical Feature Importance ---
# Extract the first linear layer weights from metadata encoder
meta_weights = meta_encoder.mlp[0].weight.data.abs().mean(dim=0).cpu().numpy()

BG='#0f1117'; AX='#1a1d27'; TC='#ecf0f1'; GC='#2c3e50'
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(BG); ax.set_facecolor(AX)

sorted_idx = np.argsort(meta_weights)
ax.barh([CLINICAL_FEATURES[i] for i in sorted_idx], meta_weights[sorted_idx],
        color='#2ecc71', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean Absolute Weight', color=TC, fontsize=12)
ax.set_title('Phase 3 -- Clinical Feature Importance', color=TC, fontsize=14, fontweight='bold')
ax.tick_params(colors=TC); [s.set_color(GC) for s in ax.spines.values()]
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, 'v3_feature_importance.png'), dpi=150, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved feature importance chart.')

In [ ]:
# --- CELL 16: Save All Results ---
preds_y = (test_probs >= youden_thresh).astype(int)
cm = confusion_matrix(test_labels, preds_y)
tn,fp,fn,tp = cm.ravel()

results_v3 = {
    'test_auc':              round(test_auc, 4),
    'default_acc':           round(accuracy_score(test_labels, (test_probs>=0.5).astype(int)), 4),
    'youden_acc':            round(accuracy_score(test_labels, preds_y), 4),
    'youden_sensitivity':    round(tp/(tp+fn), 4),
    'youden_specificity':    round(tn/(tn+fp), 4),
    'youden_threshold':      round(float(youden_thresh), 4),
    'test_n':                int(len(test_labels)),
    'clinical_features':     CLINICAL_FEATURES,
    'num_clinical_features': NUM_CLINICAL,
    'improvements': [
        'Triple Fusion: Image + Text + Clinical Metadata',
        'MetadataEncoder MLP (16 features -> 64-d)',
        'Cross-Attention (Image queries Text)',
        'Focal Loss (gamma=2.0)',
        'Mixup on embeddings (alpha=0.2)',
        'Separate LRs (BERT=1e-5, Fusion=2e-4, Meta=1e-3)',
        'ClinicalBERT last 2 layers unfrozen',
    ]
}

with open(os.path.join(SAVE_DIR, 'phase3_results.json'), 'w') as f:
    json.dump(results_v3, f, indent=2)

print('=== Phase 3 Triple Fusion Final Results ===')
for k,v in results_v3.items():
    if not isinstance(v, list):
        print(f'  {k:30s}: {v}')
print(f'Saved to: {SAVE_DIR}')

## Phase 3 Triple Fusion -- Summary

### Architecture
```
Image Encoder (Frozen DenseNet+CBAM, 1024-d) ──┐
                                                 ├── Cross-Attention ── Concat ── MLP ── Prediction
Text Encoder (ClinicalBERT, unfrozen, 768-d) ──┘                        ↑
                                                          Metadata MLP (16 → 64-d)
```

### Clinical Features Used (16 total)
| Category | Features |
|----------|----------|
| Demographics | age, gender, is_deceased |
| Vitals | heart_rate, respiratory_rate, spo2, systolic_bp, diastolic_bp, temperature_f |
| Labs | wbc, hemoglobin, hematocrit, creatinine, crp, alk_phos, albumin |

### What's New in Phase 3 vs Phase 2v2
- Added **MetadataEncoder MLP** (16 clinical features → 64-d embedding)
- Clinical metadata concatenated with cross-attention output before final classifier
- Separate learning rate for metadata encoder (1e-3, higher than BERT)

